# 🗄️ Workshop: Bau deinen eigenen WM-Experten 2026
## Notebook 3: Einführung in RAG (Retrieval-Augmented Generation)

Große Sprachmodelle (LLMs) wie ChatGPT wissen nicht alles. Sie kennen keine Echtzeitdaten und keine internen Dokumente. Wenn wir sie nach der WM 2026 fragen, fangen sie an zu halluzinieren (sie erfinden Fakten).

**RAG (Retrieval-Augmented Generation)** löst dieses Problem in drei Schritten:
1. **Suchen (Retrieval):** Bei einer Benutzerfrage suchen wir in einer eigenen Datenbank nach relevanten Textabschnitten (Chunks).
2. **Erweitern (Augment):** Wir fügen diese Textabschnitte in den Prompt für das LLM ein.
3. **Generieren (Generation):** Das LLM antwortet basierend auf dem mitgelieferten Wissen.

In diesem Notebook erstellen wir eine Vektordatenbank und füttern sie mit unseren WM-2026-Fakten.

### Lernziele:
1. Laden und Verstehen von Text-Chunks.
2. Vektorisieren von Texten (Embeddings) über das LiteLLM-Gateway.
3. Speichern und Abfragen von Vektoren in ChromaDB.


### Schritt 1: WM-Chunks laden
Die Datenpipeline hat bereits aus Spielplänen, ELO-Tabellen und Trivia-Dateien natürlichsprachliche Textblöcke (Chunks) generiert und in `data-pipeline/data/chunks.json` gespeichert. Wir laden diese nun.


In [ ]:
import json
from pathlib import Path

chunks_path = Path("../data-pipeline/data/chunks.json")
with open(chunks_path, encoding="utf-8") as f:
    chunks = json.load(f)

print(ff"✅ {len(chunks)} Text-Chunks geladen.")
print("Beispiel-Chunk:")
print(json.dumps(chunks[5], indent=2, ensure_ascii=False))


### Schritt 2: Embeddings erstellen
Ein Embedding ist eine Liste von Zahlen (ein Vektor), die die semantische Bedeutung eines Textes beschreibt. Ähnliche Texte haben ähnliche Vektoren. Wir nutzen das LiteLLM-Gateway, um diese Vektoren zu erzeugen.


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# LiteLLM Gateway einrichten (nutzt standardmäßig die Umgebungsvariablen)
LITELLM_URL = os.environ.get("OPENAI_BASE_URL", "https://litellm.fh-swf.cloud/v1")
LITELLM_KEY = os.environ.get("OPENAI_API_KEY", "dummy")

client = OpenAI(base_url=LITELLM_URL, api_key=LITELLM_KEY)

# Text in Vektor umwandeln
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=["Deutschland spielt in Gruppe E gegen Japan."]
)
vector = response.data[0].embedding

print(f"Vektordimension: {len(vector)}")
print(f"Erste 10 Werte: {vector[:10]}")


### Schritt 3: Ingestion in ChromaDB
ChromaDB ist eine schnelle, leichtgewichtige Vektordatenbank. Wir erstellen eine Collection und fügen einige Chunks mitsamt ihren Embeddings und Metadaten ein.


In [ ]:
import chromadb

CHROMA_HOST = os.environ.get("CHROMA_HOST", "localhost")
CHROMA_PORT = int(os.environ.get("CHROMA_PORT", 8000))

try:
    # Versuch, mit dem Docker-Chroma-Server zu verbinden
    chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
    print(f"ChromaDB: Verbunden mit {CHROMA_HOST}:{CHROMA_PORT}")
except Exception:
    # Fallback auf lokale, dateibasierte ChromaDB
    chroma_client = chromadb.PersistentClient(path="./chroma_db_local")
    print("ChromaDB: Nutze lokalen Speicherpfad (PersistentClient)")

collection_name = "wm2026_student"
try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)


In [ ]:
# Wir speichern die ersten 50 Chunks in unserer Datenbank
subset_chunks = chunks[:50]

texts = [c["text"] for c in subset_chunks]
ids = [c["chunk_id"] for c in subset_chunks]
metadatas = [{
    "source": c["source"],
    "data_type": c["data_type"],
    "team": c["team"]
} for c in subset_chunks]

print("Erzeuge Embeddings für Datenbank...")
emb_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)
embeddings = [item.embedding for item in emb_response.data]

print("Speichere in ChromaDB...")
collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas
)
print("Fertig!")


### Schritt 4: Semantische Suche (Retrieval)
Wir können nun Fragen stellen. Die Datenbank vergleicht den Vektor der Frage mit allen Vektoren der Chunks und gibt uns die inhaltlich ähnlichsten Texte zurück.


In [ ]:
frage = "Wer sind die Gastgeber der Weltmeisterschaft 2026?"

# 1. Frage einbetten
frage_emb = client.embeddings.create(
    model="text-embedding-3-small",
    input=[frage]
).data[0].embedding

# 2. Vektorsuche in ChromaDB
results = collection.query(
    query_embeddings=[frage_emb],
    n_results=2
)

print("Suchergebnisse für:", frage)
for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"\n[Abstand: {distance:.4f} - Quelle: {meta['source']}]")
    print(doc)


### 🧠 Aufgabe für dich:
Führe eine Suche nach der Frage: *"In welchem Stadion wird das Finale ausgetragen?"* durch. Gebe das beste Ergebnis aus.


In [ ]:
# DEINE LÖSUNG HIER:
frage_finale = "In welchem Stadion wird das Finale ausgetragen?"
frage_finale_emb = client.embeddings.create(model="text-embedding-3-small", input=[frage_finale]).data[0].embedding
res_finale = collection.query(query_embeddings=[frage_finale_emb], n_results=1)
print(res_finale["documents"][0][0])
